# AlphaSched PPO Training on Colab

Train MaskablePPO for Parallel Machine Scheduling (TWT) using Colab Pro GPU.

**Setup**: Runtime → Change runtime type → **GPU** (T4 or A100)

## 1. Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

## 2. Mount Google Drive (for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Where trained models will be saved
DRIVE_OUTPUT = '/content/drive/MyDrive/alphasched-runs'
!mkdir -p "$DRIVE_OUTPUT"

## 3. Clone repo & install

In [ ]:
import sys, subprocess

# Check Python version — need >= 3.11
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 11), (
    f"Need Python >= 3.11, got {sys.version_info.major}.{sys.version_info.minor}. "
    "Try a newer Colab runtime."
)

In [ ]:
# Clone the repository
!rm -rf /content/alpha-PPO-code
!git clone https://github.com/bwz96sco/alpha-PPO-code.git /content/alpha-PPO-code

In [ ]:
# Install the package with pip (simpler than uv on Colab)
%cd /content/alpha-PPO-code/alpha-schedule-sb3
!pip install -e . -q

In [ ]:
# Verify installation
!python -c "from alphasched.cli.train_ppo import main; print('alphasched OK')"
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

## 4. Training Configuration

Adjust these parameters before running.

In [ ]:
# ===== Training Parameters =====

# Environment
PART_NUM = 65          # Number of jobs
DIST_TYPE = "h"        # Distribution: h(igh), m(edium), l(ow)

# Training scale
TOTAL_TIMESTEPS = 2_000_000  # Increase for better results (default 200K is just a test)
NUM_ENVS = 8                 # Parallel environments
RUN_HOURS = 0                # 0 = no wall-time limit; set >0 to auto-stop

# Model architecture
POLICY_NET = "resnet"   # resnet | simconv
RESBLOCKS = 9           # Number of residual blocks (resnet only)

# PPO hyperparameters (paper Table 4 defaults)
LEARNING_RATE = 2.5e-4
BATCH_SIZE = 256
N_STEPS = 1024
N_EPOCHS = 4

print("Config ready. Run the next cell to start training.")

## 5. Train

In [ ]:
%cd /content/alpha-PPO-code/alpha-schedule-sb3

cmd = [
    "alphasched-train-ppo",
    "--part-num", str(PART_NUM),
    "--dist-type", DIST_TYPE,
    "--total-timesteps", str(TOTAL_TIMESTEPS),
    "--num-envs", str(NUM_ENVS),
    "--device", "cuda",
    "--policy-net", POLICY_NET,
    "--resblocks", str(RESBLOCKS),
    "--learning-rate", str(LEARNING_RATE),
    "--batch-size", str(BATCH_SIZE),
    "--n-steps", str(N_STEPS),
    "--n-epochs", str(N_EPOCHS),
]

if RUN_HOURS > 0:
    cmd += ["--run-hours", str(RUN_HOURS)]

print(f"Running: {' '.join(cmd)}")
!{' '.join(cmd)}

## 6. Copy results to Google Drive

In [ ]:
import glob, shutil, os

# Find the latest run directory
run_dirs = sorted(glob.glob("runs/*/"))
if run_dirs:
    latest = run_dirs[-1]
    dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest.rstrip('/')))
    shutil.copytree(latest, dest, dirs_exist_ok=True)
    print(f"Copied {latest} → {dest}")
    print(f"\nModel: {dest}/model.zip")
    print(f"Metrics: {dest}/metrics.csv")
    print(f"Config: {dest}/config.json")
else:
    print("No run directories found.")

## 7. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## 8. Quick evaluation (optional)

In [ ]:
import glob

run_dirs = sorted(glob.glob("runs/*/"))
if run_dirs:
    latest = run_dirs[-1]
    model_path = f"{latest}model.zip"
    print(f"Evaluating: {model_path}")
    !alphasched-eval-ppo --model-path "{model_path}" --part-num {PART_NUM} --dist-type {DIST_TYPE}
else:
    print("No model found. Train first.")